# Task4 教学流程：海龟交易策略与回测

## Goal

本 Notebook 从本地前复权日线出发，逐步计算高低点通道、Wilder ATR、突破信号、ATR 止损、风险定仓和绩效指标，并演示如何比较不同股票及参数。

### 当前结果概览

默认参数 `E20/X10/ATR20/S2` 在2024年以后的10只股票中有 **7/10** 取得正累计回报；跨股票中位累计回报为 **0.6%**，中位夏普比率为 **0.08**。这些结果用于教学和稳健性比较，不构成投资建议。

## Setup

### Key Assumptions

- 日线前复权价格；只做多、不加杠杆、不加仓。
- 入场通道20日、离场通道10日、ATR周期20日、止损距离2ATR。
- 通道全部向后移动1日，收盘产生通道信号，下一交易日开盘执行。
- 每笔交易计划风险为账户权益的1%，资金占用不超过100%，允许教学用小数股。
- 买卖单边综合交易成本均为0.1%，无风险利率设为0。

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "Task4").exists():
    ROOT = ROOT.parent
SCRIPTS = ROOT / "Task4" / "scripts"
sys.path.insert(0, str(SCRIPTS))

from config import DEFAULT_PARAMS, PARAMETER_SETS, UNIVERSE, parameter_label
from turtle_backtest import add_turtle_indicators, load_prices, run_backtest, summarize_backtest

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")

## Steps

### 1. 加载已保存的行情数据

In [2]:
stock = UNIVERSE[0]  # 平安银行，可替换为 UNIVERSE 中的其他股票
prices = load_prices(stock)
print(stock)
print(f"数据区间：{prices['trade_date'].min().date()} 至 {prices['trade_date'].max().date()}，共 {len(prices)} 个交易日")
prices[["trade_date", "open", "high", "low", "close", "vol"]].head()

StockSpec(ts_code='000001.SZ', stock_name='平安银行', industry='银行')
数据区间：2019-01-02 至 2026-07-10，共 1823 个交易日


,trade_date,open,high,low,close,vol
0,2019-01-02,7.2975,7.3208,7.1188,7.1421,539386.3200
1,2019-01-03,7.1343,7.2509,7.1110,7.2120,415537.9500
2,2019-01-04,7.1809,7.6317,7.1654,7.5773,1481159.0600
3,2019-01-07,7.6472,7.6550,7.4840,7.5695,865687.6600
4,2019-01-08,7.5617,7.5695,7.4762,7.5073,402388.1100


### 2. 计算前20日最高通道、前10日最低通道和ATR20

In [3]:
indicators = add_turtle_indicators(prices, DEFAULT_PARAMS)
indicators[[
    "trade_date", "close", "entry_high", "exit_low", "true_range", "atr",
    "entry_signal", "channel_exit_signal"
]].dropna().head(10)

,trade_date,close,entry_high,exit_low,true_range,atr,entry_signal,channel_exit_signal
20,2019-01-30,8.5099,8.6575,7.9037,0.2487,0.2114,False,False
21,2019-01-31,8.6264,8.6886,7.9037,0.2021,0.2109,False,False
22,2019-02-01,8.7042,8.7042,7.9736,0.2254,0.2117,False,False
23,2019-02-11,8.7119,8.7430,7.9736,0.1865,0.2104,False,False
24,2019-02-12,8.6964,8.7430,7.9736,0.2176,0.2108,False,False
25,2019-02-13,8.8440,8.7896,7.9969,0.2254,0.2115,True,False
26,2019-02-14,8.7430,8.8674,8.0591,0.1477,0.2083,False,False
27,2019-02-15,8.5099,8.8674,8.1990,0.2720,0.2115,False,False
28,2019-02-18,8.8285,8.8674,8.3700,0.3186,0.2168,False,False
29,2019-02-19,8.7586,8.8674,8.3700,0.2953,0.2208,False,False


`entry_high` 和 `exit_low` 使用 `shift(1)`，因此当日通道不包含当日最高价或最低价。真实波幅同时考虑日内波幅和相对昨收的跳空，ATR 使用 Wilder 递推平均。

### 3. 执行海龟策略和ATR风险定仓

In [4]:
result, trades = run_backtest(prices, DEFAULT_PARAMS)
result.loc[result["execution"].ne("HOLD"), [
    "trade_date", "execution", "fill_price", "entry_high", "exit_low",
    "atr", "stop_price", "exit_reason", "exposure", "strategy_equity"
]].head(12)

,trade_date,execution,fill_price,entry_high,exit_low,atr,stop_price,exit_reason,exposure,strategy_equity
26,2019-02-14,BUY,8.7819,8.8674,8.0591,0.2083,8.3589,,0.2069,99887.3734
80,2019-05-07,SELL,10.1264,11.5330,9.8777,0.3828,NaN,CHANNEL_EXIT,0.0000,103133.8296
110,2019-06-19,BUY,10.3284,9.9865,9.0150,0.3003,9.7445,,0.1745,102813.5838
144,2019-08-06,SELL,10.2880,11.4267,10.4215,0.2705,NaN,CHANNEL_EXIT,0.0000,103026.0253
149,2019-08-13,BUY,11.7801,11.8744,10.2330,0.3042,11.1711,,0.1923,102859.9772
156,2019-08-22,SELL,11.1711,11.9529,10.8770,0.3093,NaN,ATR_STOP,0.0000,101956.9437
177,2019-09-23,BUY,12.0472,12.0943,11.1362,0.2739,11.4946,,0.2185,101992.6753
215,2019-11-21,SELL,12.3692,13.6964,12.3849,0.3277,NaN,CHANNEL_EXIT,0.0000,102505.9732
234,2019-12-18,BUY,12.9032,13.0603,11.9451,0.2719,12.3515,,0.2343,102525.7758
255,2020-01-17,SELL,12.8640,13.6179,12.7226,0.2915,NaN,CHANNEL_EXIT,0.0000,102385.2171


### 4. 绘制价格、通道、止损和买卖信号

In [5]:
buys = result["execution"].eq("BUY")
sells = result["execution"].eq("SELL")

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True, gridspec_kw={"height_ratios": [2.2, 0.8]})
axes[0].plot(result["trade_date"], result["close"], color="#1f2937", linewidth=1.1, label="收盘价")
axes[0].plot(result["trade_date"], result["entry_high"], color="#2563eb", label="前20日最高通道")
axes[0].plot(result["trade_date"], result["exit_low"], color="#d97706", label="前10日最低通道")
axes[0].plot(result["trade_date"], result["stop_price"], color="#be123c", linestyle="--", label="2ATR止损线")
axes[0].scatter(result.loc[buys, "trade_date"], result.loc[buys, "fill_price"], marker="^", color="#0f766e", label="买入")
axes[0].scatter(result.loc[sells, "trade_date"], result.loc[sells, "fill_price"], marker="v", color="#be123c", label="卖出")
axes[0].set_title(f"{stock.stock_name} 海龟策略价格与交易信号")
axes[0].set_ylabel("价格（元）")
axes[0].legend(ncol=3)
axes[0].grid(alpha=0.3)

axes[1].plot(result["trade_date"], result["atr"], color="#7c3aed")
axes[1].set_title("Wilder ATR20")
axes[1].set_ylabel("ATR")
axes[1].set_xlabel("交易日期")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

C:\Users\13377\AppData\Local\Temp\ipykernel_72628\3360112297.py:21: UserWarning: Glyph 20215 (\N{CJK UNIFIED IDEOGRAPH-4EF7}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\13377\AppData\Local\Temp\ipykernel_72628\3360112297.py:21: UserWarning: Glyph 26684 (\N{CJK UNIFIED IDEOGRAPH-683C}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\13377\AppData\Local\Temp\ipykernel_72628\3360112297.py:21: UserWarning: Glyph 65288 (\N{FULLWIDTH LEFT PARENTHESIS}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\13377\AppData\Local\Temp\ipykernel_72628\3360112297.py:21: UserWarning: Glyph 20803 (\N{CJK UNIFIED IDEOGRAPH-5143}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\13377\AppData\Local\Temp\ipykernel_72628\3360112297.py:21: UserWarning: Glyph 65289 (\N{FULLWIDTH RIGHT PARENTHESIS}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\13377\AppData\Local\Temp\ipykernel_72628\3360112297.py:21: UserWarning: Glyph 24179 (

### 5. 计算累计回报、MDD、夏普比率等指标

In [6]:
metrics = summarize_backtest(result, trades)
pd.Series(metrics, name=parameter_label(DEFAULT_PARAMS))

observations             1823.0000
cumulative_return          -0.0535
annualized_return          -0.0076
annualized_volatility       0.0406
sharpe_ratio               -0.1669
max_drawdown               -0.1360
benchmark_return            0.4632
excess_return              -0.5166
buy_count                  34.0000
sell_count                 34.0000
completed_trades           34.0000
win_rate                    0.2941
average_holding_days       17.7353
holding_ratio               0.3308
average_exposure            0.0827
total_transaction_cost   1565.7481
total_turnover             16.1299
atr_stop_exits             14.0000
channel_exits              20.0000
open_position_at_end        0.0000
Name: E20/X10/ATR20/S2, dtype: float64

### 6. 比较不同通道和止损参数的样本外表现

In [7]:
comparison = pd.read_csv(ROOT / "Task4" / "outputs" / "parameter_comparison.csv")
stock_oos = comparison.loc[
    (comparison["ts_code"] == stock.ts_code) &
    (comparison["sample_period"] == "out_of_sample"),
    ["parameter", "cumulative_return", "max_drawdown", "sharpe_ratio", "completed_trades", "win_rate"]
].sort_values("sharpe_ratio", ascending=False)
stock_oos

,parameter,cumulative_return,max_drawdown,sharpe_ratio,completed_trades,win_rate
11,E20/X10/ATR20/S3,0.0191,-0.0412,0.2491,11,0.3636
2,E10/X5/ATR14/S1.5,-0.0044,-0.0943,0.0073,21,0.1429
8,E20/X10/ATR20/S2,-0.0027,-0.0535,-0.0016,12,0.2500
5,E20/X10/ATR20/S1.5,-0.0206,-0.0792,-0.1202,14,0.2143
17,E55/X20/ATR20/S2,-0.0224,-0.0522,-0.2180,6,0.3333
14,E40/X20/ATR20/S2,-0.0525,-0.0733,-0.5092,10,0.2000


## Checks

下面独立重算关键公式，并检查信号时序、资金恒等式和指标范围。

In [8]:
expected_upper = prices["high"].rolling(DEFAULT_PARAMS.entry_window).max().shift(1)
expected_lower = prices["low"].rolling(DEFAULT_PARAMS.exit_window).min().shift(1)
previous_close = prices["close"].shift(1)
expected_tr = pd.concat([
    prices["high"] - prices["low"],
    (prices["high"] - previous_close).abs(),
    (prices["low"] - previous_close).abs(),
], axis=1).max(axis=1)

np.testing.assert_allclose(result["entry_high"], expected_upper, equal_nan=True)
np.testing.assert_allclose(result["exit_low"], expected_lower, equal_nan=True)
np.testing.assert_allclose(result["true_range"], expected_tr, equal_nan=True)
np.testing.assert_allclose(result["strategy_equity"], result["cash"] + result["shares"] * result["close"], rtol=1e-10)
assert result.loc[:DEFAULT_PARAMS.entry_window - 1, "execution"].eq("HOLD").all()
assert result["drawdown"].between(-1, 0).all()
assert (result["transaction_cost"] >= 0).all()
assert set(result.loc[result["execution"].eq("SELL"), "exit_reason"]) <= {"ATR_STOP", "CHANNEL_EXIT"}

print("检查通过：通道错位、TR公式、资金恒等式、回撤范围和退出原因均符合预期。")

检查通过：通道错位、TR公式、资金恒等式、回撤范围和退出原因均符合预期。


## Next Steps

1. 修改 `stock = UNIVERSE[0]`，比较不同股票和行业。
2. 修改 `PARAMETER_SETS`，观察短通道、长通道和不同ATR止损倍数的变化。
3. 优先比较2024年以后的样本外结果，同时检查最大回撤、交易次数和基准收益。
4. 进一步加入100股整手、涨跌停、滑点、停牌和组合级风险预算，使模型更接近A股实盘。